# Manufacturing Output Forecasting

## Project Overview

Forecasts **daily manufacturing output** (units produced) over a 14-day horizon. Synthetic data spans 2 years with weekly shift patterns, maintenance shutdowns, and seasonal demand-driven production.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily production output, predict the next 14 days. Manufacturers need output forecasts for raw material procurement, delivery commitments, and workforce planning.

## Why This Project Matters

Manufacturing is capital-intensive. Unplanned downtime costs thousands per hour. Accurate production forecasts enable just-in-time material ordering, realistic delivery promises, and proactive maintenance scheduling.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily production output
- Weekly pattern (Mon-Fri production, weekend reduced)
- Planned maintenance shutdowns (quarterly)
- Seasonal demand variation
- Gradual efficiency improvement trend

| Column | Description |
|--------|-------------|
| `date` | Date |
| `units_produced` | Daily production output (units) |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'units_produced'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

base = 5000 + 0.5 * t  # efficiency improvement
weekly = np.zeros(N_POINTS)
for i in range(N_POINTS):
    dow = dates[i].dayofweek
    if dow <= 4: weekly[i] = 500  # weekday production
    elif dow == 5: weekly[i] = -1500  # Saturday reduced
    else: weekly[i] = -3000  # Sunday minimal

seasonal = 400 * np.sin(2 * np.pi * (t - 60) / 365)  # demand-driven

# Quarterly maintenance shutdowns (3-day reductions)
maintenance = np.zeros(N_POINTS)
for q_start in [80, 170, 260, 350, 445, 535, 625, 715]:
    if q_start + 3 <= N_POINTS:
        for j in range(3):
            maintenance[q_start + j] = -3000

noise = np.random.normal(0, 200, N_POINTS)
units_produced = base + weekly + seasonal + maintenance + noise
units_produced = np.maximum(units_produced, 500).round(0).astype(int)

df = pd.DataFrame({'date': dates, 'units_produced': units_produced})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,units_produced
0,2022-01-01,3256
1,2022-01-02,1633
2,2022-01-03,5294
3,2022-01-04,5474
4,2022-01-05,5127


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('units_produced Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("units_produced Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

units_produced Statistics:
count     730.000000
mean     4795.942466
std      1408.891180
min       500.000000
25%      3774.250000
50%      5423.500000
75%      5811.500000
max      6671.000000
Name: units_produced, dtype: float64

CV: 0.294
Skewness: -1.018


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:     2180.4 | RMSE:     2582.2 | MAPE: 44.53%
Seasonal Naive (7)             | MAE:     1049.1 | RMSE:     1703.8 | MAPE: 33.09%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                          Adjusted R-Squared  R-Squared         RMSE  Time Taken
Model                                                                           
KernelRidge                       124.773870  -8.521067  4870.570796    0.058568
MLPRegressor                      108.229051  -7.248389  4533.374607    0.268035
LinearSVR                          92.709197  -6.054554  4192.487941    0.007492
QuantileRegressor                  18.548108  -0.349854  1833.920605    0.052064
SVR                                17.866179  -0.297398  1797.933932    0.023173
DummyRegressor                     14.479477  -0.036883  1607.318354    0.006152
NuSVR                              14.059924  -0.004610  1582.106462    0.018114
GaussianProcessRegressor            8.210408   0.445353  1175.561322    0.040577
GammaRegressor                      3.882634   0.778259   743.293149    2.635885
ElasticNetCV                        3.844935   0.781159   738.416731    0.054025


## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: catboost
FLAML (catboost)               | MAE:      351.4 | RMSE:      514.7 | MAPE: 28.11%
FLAML Test (catboost)          | MAE:      472.5 | RMSE:      594.0 | MAPE: 14.36%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:      433.8 | RMSE:      805.1 | MAPE: 16.01%
SF AutoETS                     | MAE:     1193.9 | RMSE:     1227.2 | MAPE: 33.46%
SF SeasonalNaive               | MAE:      699.8 | RMSE:     1292.9 | MAPE: 29.81%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
                model         MAE        RMSE      MAPE
     FLAML (catboost)  351.416409  514.744291 28.113741
         SF AutoARIMA  433.809753  805.109581 16.007091
FLAML Test (catboost)  472.492786  594.001659 14.356019
     SF SeasonalNaive  699.785706 1292.928556 29.805745
   Seasonal Naive (7) 1049.071429 1703.814984 33.091960
           SF AutoETS 1193.916748 1227.213918 33.464594
   Naive (Last Value) 2180.357143 2582.152106 44.532852

Top 3:
                model        MAE       RMSE      MAPE
     FLAML (catboost) 351.416409 514.744291 28.113741
         SF AutoARIMA 433.809753 805.109581 16.007091
FLAML Test (catboost) 472.492786 594.001659 14.356019


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: -51.03, Std: 591.81


## Interpretation and Business Insight

- Manufacturing output follows strict weekday production schedules
- Quarterly maintenance shutdowns create predictable drops
- Efficiency improvements drive a slow upward trend
- Seasonal demand variation modulates production levels
- Weekend operations are minimal (maintenance and cleanup)

## Limitations

1. Synthetic — real output depends on machine uptime, material supply, quality
2. No machine health/sensor data
3. No raw material availability constraints
4. No quality rejection rates
5. No order backlog or demand signal

## How to Improve This Project

1. Add machine sensor data for predictive maintenance
2. Include raw material inventory levels
3. Model quality rejection rates
4. Add order backlog as demand signal
5. Use shift-level data for finer granularity

## Production Considerations

- Daily and weekly production planning
- Integration with MES (manufacturing execution system)
- Raw material procurement automation
- Delivery date commitment optimization

## Common Mistakes

1. Not accounting for planned maintenance shutdowns
2. Ignoring machine degradation between maintenance cycles
3. Using daily targets without considering shift constraints
4. Not linking production forecasts to demand forecasts

## Mini Challenge / Exercises

1. Add synthetic machine health data and predict downtime
2. Model the efficiency improvement trend — is it linear?
3. Detect unplanned downtime events in residuals
4. Build a raw material requirement forecast from output forecast

## Final Summary / Key Takeaways

1. Manufacturing output has predictable weekly and maintenance patterns
2. Efficiency improvements create a slow upward trend
3. Planned maintenance causes regular predictable drops
4. Machine sensor data is the key missing predictor
5. Production forecasts must link to demand and material availability

In [18]:
import json
metrics = {
    'project': 'Manufacturing Output Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Manufacturing Output Forecasting — Complete ===")

Metrics saved.

=== Manufacturing Output Forecasting — Complete ===
